# Decimation Encoding Analysis Orchestrator

Runs a Poisson GLM encoding model on random subsets (portions) of each patient's **quality-filtered** words.
Filtration is based on `{patient}_word_df_filtered.csv` (same dataset as `word_level_duration_cv_filtered_speech`).
Subsampling (decimation) is applied after filtration.

**Two-phase submission:**
1. **Phase 1** — submit full-data jobs (portion=1.0, sample=0) per patient. These run inner-CV alpha tuning and save `{patient}_best_alphas.npy`.
2. **Phase 2** — submit all other (portion, sample) jobs with SLURM `--dependency=afterok` on the full-data job and `--fixed-alpha-path` pointing to the saved alphas. No inner CV for portion runs.

**Workflow:**
1. Run config + helper cells
2. Run the **Compute filtered word indices** cell
3. Run **Phase 1: submit full-data jobs** cell
4. Run **Phase 2: submit portion jobs** cell immediately — SLURM holds them until Phase 1 completes
5. Re-run status cells periodically to check progress
6. Run assembly cells once all jobs complete

In [1]:
from pathlib import Path
import subprocess
import json

import dill as pickle
import numpy as np
import pandas as pd
import torch

In [8]:
# ── User-facing settings ───────────────────────────────────────────────────────
PROJ_DIR    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT    = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/decimation_analysis/decimation_glm_worker.py')
PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS      = None   # None => scan all Y* patient folders under vad_new
FORCE_RESUBMIT = False

# Decimation grid — must match original analysis
# portions[0] must be 1.0 (full data); its samples[0] must be 1
PORTIONS  = [1.0,  0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_SAMPLES = [1,   100,  20,   10,  8,   7,   6,   5,   4,   3,   2,   2  ]

# Model settings
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100
OUTER_SPLITS     = 5
INNER_SPLITS     = 5
N_ALPHAS         = 30
ALPHA_LOW        = -3.0
ALPHA_HIGH       = 3.0
N_SHUFFLES       = 50


# Compute mode — set False to submit CPU-only jobs (no GPU slot required)
USE_GPU = False

# SLURM settings
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    MEM_FULL  = '64G'
    TIME_FULL = '24:00:00'
    N_PERM_JOBS      = CPUS #* 2
else:
    PARTITION = 'Universal'
    GRES      = ''       # no GPU
    CPUS      = 16
    MEM_FULL  = '32G'
    TIME_FULL = '72:00:00'
    N_PERM_JOBS      = CPUS #* 2
# Portion jobs use adaptive mem/time from compute_resources() based on n_words estimate

RUN_NAME            = 'decimation_cv'
GLOBAL_LOGS_DIR     = VAD_ROOT / f'{RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR  = VAD_ROOT / f'{RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('worker:     ', WORKER_PATH)
print('logs dir:   ', GLOBAL_LOGS_DIR)
print('scripts dir:', GLOBAL_SCRIPTS_DIR)
print('portions:   ', PORTIONS)
print('n_samples:  ', N_SAMPLES)
print(f'compute:     {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:      /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/decimation_analysis/decimation_glm_worker.py
logs dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/decimation_cv_slurm_logs
scripts dir: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/decimation_cv_slurm_scripts
portions:    [1.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
n_samples:   [1, 100, 20, 10, 8, 7, 6, 5, 4, 3, 2, 2]
compute:     CPU | partition=Universal


## Compute filtered word indices

Reads `{patient}_word_df_filtered.csv` and saves the `original_word_idx` column as
`{patient}_filtered_word_idx.npy`. These indices select quality-filtered words from the
full word arrays before any decimation subsampling is applied.

In [3]:
def compute_filtered_indices(patient: str, out_dir: Path) -> 'Path | None':
    filtered_csv = VAD_ROOT / patient / f'{patient}_word_df_filtered.csv'
    idx_path     = out_dir / f'{patient}_filtered_word_idx.npy'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return None

    filtered_df = pd.read_csv(filtered_csv)
    word_idx    = filtered_df['original_word_idx'].values.astype(np.int64)
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(idx_path, word_idx)
    print(f'  [{patient}] {len(word_idx):,} filtered words -> {idx_path.name}')
    return idx_path


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

patient_idx_paths = {}
for patient in all_patients:
    out_dir  = VAD_ROOT / patient / 'encoding' / RUN_NAME
    idx_path = compute_filtered_indices(patient, out_dir)
    if idx_path is not None:
        patient_idx_paths[patient] = idx_path

print(f'\nfiltered indices ready: {len(patient_idx_paths)} / {len(all_patients)} patients')

  [YEY] 40,351 filtered words -> YEY_filtered_word_idx.npy
  [YEZ] 71,809 filtered words -> YEZ_filtered_word_idx.npy
  [YFA] 521,360 filtered words -> YFA_filtered_word_idx.npy
  [YFB] 128,315 filtered words -> YFB_filtered_word_idx.npy
  [YFC] 292,428 filtered words -> YFC_filtered_word_idx.npy
  [YFD] 388,734 filtered words -> YFD_filtered_word_idx.npy
  [YFE] 199,717 filtered words -> YFE_filtered_word_idx.npy
  [YFF] 228,795 filtered words -> YFF_filtered_word_idx.npy
  [YFG] 48,567 filtered words -> YFG_filtered_word_idx.npy
  [YFI] 126,297 filtered words -> YFI_filtered_word_idx.npy
  [YFK] 372,685 filtered words -> YFK_filtered_word_idx.npy
  [YFL] 90,616 filtered words -> YFL_filtered_word_idx.npy
  [YFM] 216,077 filtered words -> YFM_filtered_word_idx.npy
  [YFN] 286,018 filtered words -> YFN_filtered_word_idx.npy
  [YFP] 375,545 filtered words -> YFP_filtered_word_idx.npy
  [YFQ] 171,650 filtered words -> YFQ_filtered_word_idx.npy
  [YFR] 205,562 filtered words -> YFR_filter

In [4]:
# ── Helper functions ───────────────────────────────────────────────────────────

def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def compute_resources(n_words: int) -> dict:
    """
    Return adaptive SLURM gres/mem/time for a portion job based on estimated word count.

    GPU mode uses MPS fractions; CPU mode scales time up and omits the GPU requirement.
    Scaling rationale (N_SHUFFLES=50, OUTER_SPLITS=5, no inner CV):
      Total LBFGS fits ≈ outer_splits * (1 + n_shuffles) = 255.
      Fit cost scales roughly linearly with n_words.
    """
    if USE_GPU:
        if n_words >= 20_000:
            return dict(gres='mps:l40:100', mem='32G', time='24:00:00')
        elif n_words >= 10_000:
            return dict(gres='mps:l40:50', mem='16G', time='12:00:00')
        elif n_words >= 3_000:
            return dict(gres='mps:l40:33', mem='12G',  time='8:00:00')
        else:
            return dict(gres='mps:l40:10', mem='8G',  time='4:00:00')
    else:
        if n_words >= 20_000:
            return dict(gres='', mem='16G', time='48:00:00')
        elif n_words >= 10_000:
            return dict(gres='', mem='12G', time='24:00:00')
        elif n_words >= 3_000:
            return dict(gres='', mem='8G',  time='16:00:00')
        else:
            return dict(gres='', mem='4G',  time='8:00:00')


def resolve_patient_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])
    out_dir       = patient_root / 'encoding' / RUN_NAME
    alpha_path    = out_dir / f'{patient}_best_alphas.npy'
    word_idx_path = patient_idx_paths.get(patient)
    # load word count from the idx file — it's a tiny int array so this is cheap
    n_filtered_words = (len(np.load(word_idx_path))
                        if word_idx_path is not None and Path(word_idx_path).exists()
                        else None)
    return dict(
        patient=patient,
        patient_root=patient_root,
        embeddings_path=embeddings_path,
        counts_path=counts_path,
        durations_path=durations_path,
        word_idx_path=word_idx_path,
        n_filtered_words=n_filtered_words,
        out_dir=out_dir,
        alpha_path=alpha_path,
    )


def job_tag(portion, sample):
    return f'portion{portion}_sample{sample}'


def job_paths(info: dict, portion: float, sample: int) -> dict:
    patient = info['patient']
    tag     = job_tag(portion, sample)
    d       = info['out_dir']
    return dict(
        success = d / f'{patient}_{tag}_SUCCESS',
        error   = d / f'{patient}_{tag}_error.txt',
        pkl     = d / f'{patient}_{tag}_encoding_results_cv.pkl',
        tar     = d / f'{patient}_{tag}_encoding_models_cv.tar',
        meta    = d / f'{patient}_{tag}_meta.json',
    )


def build_patient_status(patients=None) -> pd.DataFrame:
    if patients is None:
        patients = sorted(p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y'))
    rows = []
    for patient in patients:
        info = resolve_patient_paths(patient)
        row = dict(
            patient=patient,
            has_embeddings=info['embeddings_path'] is not None,
            has_counts=info['counts_path'] is not None,
            has_durations=info['durations_path'] is not None,
            has_word_idx=info['word_idx_path'] is not None,
            n_filtered_words=info['n_filtered_words'],
            has_best_alphas=info['alpha_path'].exists(),
        )
        row['ready_inputs'] = (row['has_embeddings'] and row['has_counts']
                               and row['has_durations'] and row['has_word_idx'])
        full_paths = job_paths(info, 1.0, 0)
        row['full_done']  = full_paths['success'].exists()
        row['full_error'] = full_paths['error'].exists()
        done_jobs = sum(
            1 for p_idx, p in enumerate(PORTIONS)
            for s in range(N_SAMPLES[p_idx])
            if job_paths(info, p, s)['success'].exists()
        )
        row['jobs_done']  = done_jobs
        row['jobs_total'] = sum(N_SAMPLES)
        rows.append(row)
    return pd.DataFrame(rows).sort_values('patient').reset_index(drop=True)


def build_job_status(patients=None) -> pd.DataFrame:
    """Flat table: one row per (patient, portion, sample)."""
    if patients is None:
        patients = sorted(p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y'))
    rows = []
    for patient in patients:
        info = resolve_patient_paths(patient)
        if not (info['embeddings_path'] and info['counts_path']
                and info['durations_path'] and info['word_idx_path']):
            continue
        for p_idx, portion in enumerate(PORTIONS):
            for sample in range(N_SAMPLES[p_idx]):
                paths = job_paths(info, portion, sample)
                rows.append(dict(
                    patient=patient,
                    portion=portion,
                    sample=sample,
                    is_full=(portion == 1.0 and sample == 0),
                    done=(
                        paths['success'].exists() and
                        paths['meta'].exists() and
                        json.loads(paths['meta'].read_text()).get('n_shuffles', 0) > 0
                    ),
                    error=paths['error'].exists(),
                    has_pkl=paths['pkl'].exists(),
                ))
    return pd.DataFrame(rows)


print('helpers defined')

helpers defined


In [5]:
# ── Patient status overview ────────────────────────────────────────────────────
patient_status = build_patient_status(PATIENTS)
print('patients:', len(patient_status))
print('ready inputs (incl. filter):', int(patient_status['ready_inputs'].sum()))
print('full jobs done:', int(patient_status['full_done'].sum()))
print('have best_alphas:', int(patient_status['has_best_alphas'].sum()))
display(patient_status[['patient', 'has_word_idx', 'n_filtered_words', 'ready_inputs',
                         'full_done', 'full_error', 'has_best_alphas', 'jobs_done', 'jobs_total']])

patients: 21
ready inputs (incl. filter): 21
full jobs done: 21
have best_alphas: 21


,patient,has_word_idx,n_filtered_words,ready_inputs,full_done,full_error,has_best_alphas,jobs_done,jobs_total
0,YEY,True,40351,True,True,True,True,168,168
1,YEZ,True,71809,True,True,True,True,164,168
2,YFA,True,521360,True,True,False,True,127,168
3,YFB,True,128315,True,True,True,True,153,168
4,YFC,True,292428,True,True,True,True,139,168
5,YFD,True,388734,True,True,True,True,132,168
6,YFE,True,199717,True,True,True,True,145,168
7,YFF,True,228795,True,True,True,True,140,168
8,YFG,True,48567,True,True,True,True,168,168
9,YFI,True,126297,True,True,True,True,152,168


## Phase 1: Submit full-data jobs

One job per patient with `portion=1.0, sample=0`. Runs inner-CV alpha tuning over the full filtered word set and saves `{patient}_best_alphas.npy` for use by all portion jobs.

In [6]:
full_job_ids = {}   # {patient: slurm_job_id}  — used by Phase 2 for --dependency
submitted_full = []
skipped_full   = []

for _, row in patient_status.iterrows():
    patient = row['patient']
    if not row['ready_inputs']:
        continue

    info  = resolve_patient_paths(patient)
    paths = job_paths(info, 1.0, 0)

    if paths['success'].exists() and not FORCE_RESUBMIT:
        skipped_full.append(patient)
        full_job_ids[patient] = 'already_done'
        continue

    info['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = f"rm -f {paths['success']} {paths['error']}" if FORCE_RESUBMIT else 'true'

    cmd = (
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {info["out_dir"]} '
        f'--portion 1.0 --sample 0 '
        f'--word-idx-path {info["word_idx_path"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA} '
        f'--outer-splits {OUTER_SPLITS} '
        f'--inner-splits {INNER_SPLITS} '
        f'--n-alphas {N_ALPHAS} '
        f'--alpha-low {ALPHA_LOW} '
        f'--alpha-high {ALPHA_HIGH} '
        f'--n-shuffles {N_SHUFFLES} '
        f'--n-perm-jobs {N_PERM_JOBS} '
        f'--embeddings-path {info["embeddings_path"]} '
        f'--counts-path {info["counts_path"]} '
        f'--durations-path {info["durations_path"]}'
    )

    gres_lines = [f'#SBATCH --gres={GRES}'] if GRES else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name=dec_full_{patient}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --mem={MEM_FULL}',
        f'#SBATCH --time={TIME_FULL}',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_full_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_full_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient} | FULL DATA (filtered)"',
        reset_line,
        cmd,
        'echo "END: $(date)"',
    ]

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_full.sbatch'
    sbatch_path.write_text('\n'.join(lines))

    try:
        res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        job_id = res.stdout.strip().split()[-1]
        full_job_ids[patient] = job_id
        submitted_full.append((patient, job_id))
        print(f'submitted full: {patient} -> job {job_id}')
    except subprocess.CalledProcessError as exc:
        print(f'FAILED SUBMISSION: {patient}\n{exc.stderr}')

print(f'\nsubmitted={len(submitted_full)}, skipped={len(skipped_full)}')
print('full_job_ids:', full_job_ids)


submitted=0, skipped=21
full_job_ids: {'YEY': 'already_done', 'YEZ': 'already_done', 'YFA': 'already_done', 'YFB': 'already_done', 'YFC': 'already_done', 'YFD': 'already_done', 'YFE': 'already_done', 'YFF': 'already_done', 'YFG': 'already_done', 'YFI': 'already_done', 'YFK': 'already_done', 'YFL': 'already_done', 'YFM': 'already_done', 'YFN': 'already_done', 'YFP': 'already_done', 'YFQ': 'already_done', 'YFR': 'already_done', 'YFS': 'already_done', 'YFT': 'already_done', 'YFU': 'already_done', 'YFV': 'already_done'}


## Phase 2: Submit portion jobs

One job per (patient, portion, sample) excluding the full-data run. Each job:
- Uses the same `--word-idx-path` (same filtered set); subsampling is applied *within* that set
- Uses `--fixed-alpha-path {patient}_best_alphas.npy` (skips inner CV)
- Has `--dependency=afterok:{full_job_id}` if the full job hasn't finished yet

Run this cell immediately after Phase 1 — SLURM holds the jobs until the dependency completes.

In [9]:
submitted_portions = []

for _, row in patient_status.iterrows():
    patient = row['patient']
    if not row['ready_inputs']:
        continue

    info = resolve_patient_paths(patient)
    info['out_dir'].mkdir(parents=True, exist_ok=True)

    full_done = job_paths(info, 1.0, 0)['success'].exists()
    full_jid  = full_job_ids.get(patient, 'already_done')
    dep_str   = ''
    if not full_done and full_jid not in ('already_done', ''):
        dep_str = f'#SBATCH --dependency=afterok:{full_jid}'

    # n_filtered_words is read from the saved idx file; fall back to 2000 if unavailable
    n_filtered = info['n_filtered_words'] or 2000

    for p_idx, portion in enumerate(PORTIONS):
        if portion == 1.0:
            continue   # full run handled in Phase 1
        for sample in range(N_SAMPLES[p_idx]):
            paths = job_paths(info, portion, sample)

            if paths['success'].exists() and not FORCE_RESUBMIT:
                continue

            # adaptive mem/time based on estimated word count for this (portion, sample)
            n_words_est = max(1, int(round(portion * n_filtered)))
            res         = compute_resources(n_words_est)

            reset_line = f"rm -f {paths['success']} {paths['error']}" if FORCE_RESUBMIT else 'true'

            cmd = (
                f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {info["out_dir"]} '
                f'--portion {portion} --sample {sample} '
                f'--word-idx-path {info["word_idx_path"]} '
                f'--fixed-alpha-path {info["alpha_path"]} '
                f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
                f'--gpt2-layer {GPT2_LAYER} '
                f'--n-pca {N_PCA} '
                f'--outer-splits {OUTER_SPLITS} '
                f'--inner-splits {INNER_SPLITS} '
                f'--n-alphas {N_ALPHAS} '
                f'--alpha-low {ALPHA_LOW} '
                f'--alpha-high {ALPHA_HIGH} '
                f'--n-shuffles {N_SHUFFLES} '
                f'--n-perm-jobs {N_PERM_JOBS} '
                f'--embeddings-path {info["embeddings_path"]} '
                f'--counts-path {info["counts_path"]} '
                f'--durations-path {info["durations_path"]}'
            )

            tag = job_tag(portion, sample)
            gres_lines = [f'#SBATCH --gres={res["gres"]}'] if res['gres'] else []
            lines = [
                '#!/bin/bash',
                f'#SBATCH --job-name=dec_{patient}_p{portion}',
                f'#SBATCH --partition={PARTITION}',
                *gres_lines,
                f'#SBATCH --cpus-per-task={CPUS}',
                f'#SBATCH --qos=big_batch_tier',
                f'#SBATCH --exclude=guppy-1',
                f'#SBATCH --mem={res["mem"]}',
                f'#SBATCH --time={res["time"]}',
                f'# estimated n_words={n_words_est} (portion={portion} x n_filtered={n_filtered})',
                f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_{tag}_%j.out',
                f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_{tag}_%j.err',
            ]
            if dep_str:
                lines.append(dep_str)
            lines += [
                '',
                'set -eo pipefail',
                CONDA_ACTIVATE,
                'echo "HOSTNAME: $(hostname)"',
                'echo "START: $(date)"',
                f'echo "PATIENT: {patient} | portion={portion} sample={sample} (~{n_words_est} words)"',
                reset_line,
                cmd,
                'echo "END: $(date)"',
            ]

            sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_{tag}.sbatch'
            sbatch_path.write_text('\n'.join(lines))

            try:
                res_sub = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                job_id  = res_sub.stdout.strip().split()[-1]
                submitted_portions.append((patient, portion, sample, job_id))
            except subprocess.CalledProcessError as exc:
                print(f'FAILED: {patient} {tag}\n{exc.stderr}')

print(f'submitted {len(submitted_portions)} portion jobs')

submitted 504 portion jobs


In [ ]:
# ── Check all job status ───────────────────────────────────────────────────────
job_df = build_job_status(PATIENTS)
total  = len(job_df)
done   = job_df['done'].sum()
erred  = job_df['error'].sum()
print(f'total jobs: {total}  |  done: {done}  |  errors: {erred}  |  pending: {total - done - erred}')

by_patient = job_df.groupby('patient').agg(
    total=('done', 'count'),
    done=('done', 'sum'),
    errors=('error', 'sum'),
).reset_index()
display(by_patient)

In [ ]:
# ── Inspect errors ─────────────────────────────────────────────────────────────
error_rows = job_df[job_df['error']].head(10)
print(f'jobs with error files: {len(job_df[job_df["error"]])}')
for _, row in error_rows.iterrows():
    patient = row['patient']
    info    = resolve_patient_paths(patient)
    paths   = job_paths(info, row['portion'], row['sample'])
    print('=' * 80)
    print(f"{patient}  portion={row['portion']}  sample={row['sample']}")
    print(paths['error'].read_text()[:2000])

## Assemble results

Load all completed PKL files, concatenate into one DataFrame, and assemble per-patient model tars.

In [ ]:
# ── Assemble stats DataFrame ───────────────────────────────────────────────────
records = []
loaded  = []

for _, row in job_df[job_df['has_pkl']].iterrows():
    patient = row['patient']
    info    = resolve_patient_paths(patient)
    paths   = job_paths(info, row['portion'], row['sample'])
    with open(paths['pkl'], 'rb') as f:
        df = pickle.load(f)
    df['patient'] = patient
    records.append(df)
    loaded.append((patient, row['portion'], row['sample']))

if records:
    all_results = pd.concat(records, ignore_index=True)
    print(f'loaded {len(records)} result files across {len(set(p for p,_,_ in loaded))} patients')
    print(f'total rows: {len(all_results)}')
    display(all_results.head())

    summary_rows = all_results[all_results['is_summary'] == True].copy()
    if 'pseudo_r2_mean' in summary_rows.columns:
        overview = summary_rows.groupby(['patient', 'portion']).agg(
            n_neurons=('neuron_idx', 'nunique'),
            n_samples=('sample', 'nunique'),
            r2_mean=('pseudo_r2_mean', 'mean'),
            pearson_mean=('pearson_corr_mean', 'mean'),
        ).reset_index()
        display(overview.sort_values(['patient', 'portion']))
else:
    print('No completed result PKLs found yet.')

In [ ]:
# ── Save assembled stats DataFrame ────────────────────────────────────────────
if records:
    out_path = VAD_ROOT / f'{RUN_NAME}_results_all.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'saved: {out_path}')

In [ ]:
# ── Assemble per-patient model tars ───────────────────────────────────────────
# Merges all (portion, sample, fold) model state_dicts into one tar per patient
# with keys: 'portion_{p}_sample_{s}_fold_{f}' — matching the original decimation format

patients_to_assemble = list(set(p for p,_,_ in loaded))

for patient in patients_to_assemble:
    info      = resolve_patient_paths(patient)
    pt_models = {}

    patient_rows = job_df[(job_df['patient'] == patient) & (job_df['has_pkl'])]
    for _, row in patient_rows.iterrows():
        paths    = job_paths(info, row['portion'], row['sample'])
        tar_path = paths['tar']
        if not tar_path.exists():
            continue
        models = torch.load(tar_path, map_location='cpu')
        pt_models.update(models)   # keys already formatted as portion_X_sample_Y_fold_Z

    out_tar = info['out_dir'] / f'{patient}_decimation_models_all.tar'
    torch.save(pt_models, out_tar)
    print(f'assembled {len(pt_models)} models -> {out_tar}')